Règle de gestion : 
- Liaison des store via les 4 première lettres du sale_id
- Calcul des clients les plus dépensiers basé uniquement sur ceux qui ont fournit des informations (email ou identity)

# Chargement des datasets  

## Dataset vente salesforces

In [116]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = SparkSession.builder.master("local[3]").appName("example").getOrCreate()

df_sales1 = spark.read.csv("Data/202401_sfcc_sales.csv", header=True, inferSchema=True)
df_sales2 = spark.read.csv("Data/202402_sfcc_sales.csv", header=True, inferSchema=True)
df_sales3 = spark.read.csv("Data/202403_sfcc_sales.csv", header=True, inferSchema=True)
df_sales4 = spark.read.csv("Data/202404_sfcc_sales.csv", header=True, inferSchema=True)
df_sales5 = spark.read.csv("Data/202405_sfcc_sales.csv", header=True, inferSchema=True)
df_sales6 = spark.read.csv("Data/202406_sfcc_sales.csv", header=True, inferSchema=True)
df_sales7 = spark.read.csv("Data/202407_sfcc_sales.csv", header=True, inferSchema=True)
df_sales8 = spark.read.csv("Data/202408_sfcc_sales.csv", header=True, inferSchema=True)
df_sales9 = spark.read.csv("Data/202409_sfcc_sales.csv", header=True, inferSchema=True)
df_sales10 = spark.read.csv("Data/202410_sfcc_sales.csv", header=True, inferSchema=True)
df_sales11 = spark.read.csv("Data/202411_sfcc_sales.csv", header=True, inferSchema=True)
df_sales12 = spark.read.csv("Data/202412_sfcc_sales.csv", header=True, inferSchema=True)

# Define the schema to be used for all DataFrames
schema = {
    "sale_id": "string",
    "transaction_date": "timestamp",
    "product_id": "string",
    "customer_id": "int",
    "customer_last_name": "string",
    "customer_first_name": "string",
    "customer_email": "string",
    "customer_address": "string",
    "customer_phone": "string",
    "email_optin": "boolean",
    "sms_optin": "boolean"
}

# Function to cast DataFrame columns to the specified schema
def cast_df(df, schema):
    for col_name, col_type in schema.items():
        df = df.withColumn(col_name, col(col_name).cast(col_type))
    return df

# Cast all DataFrames to the same schema
df_sales1 = cast_df(df_sales1, schema)
df_sales2 = cast_df(df_sales2, schema)
df_sales3 = cast_df(df_sales3, schema)
df_sales4 = cast_df(df_sales4, schema)
df_sales5 = cast_df(df_sales5, schema)
df_sales6 = cast_df(df_sales6, schema)
df_sales7 = cast_df(df_sales7, schema)
df_sales8 = cast_df(df_sales8, schema)
df_sales9 = cast_df(df_sales9, schema)
df_sales10 = cast_df(df_sales10, schema)
df_sales11 = cast_df(df_sales11, schema)
df_sales12 = cast_df(df_sales12, schema)

# Union all DataFrames
df_sales = df_sales1.union(df_sales2).union(df_sales3).union(df_sales4).union(df_sales5).union(df_sales6).union(df_sales7).union(df_sales8).union(df_sales9).union(df_sales10).union(df_sales11).union(df_sales12)
df_sales.orderBy(df_sales["transaction_date"].desc()).show()


+-------+-------------------+----------+-----------+------------------+-------------------+--------------------+--------------------+--------------+-----------+---------+
|sale_id|   transaction_date|product_id|customer_id|customer_last_name|customer_first_name|      customer_email|    customer_address|customer_phone|email_optin|sms_optin|
+-------+-------------------+----------+-----------+------------------+-------------------+--------------------+--------------------+--------------+-----------+---------+
| S477v4|2024-12-31 00:00:00|   P594797|    4564563|              Blin|             Maxime|maxime.blin@hotma...|45 Rue Leyteire, ...|     765164710|      false|     true|
| S5x9v7|2024-12-30 00:00:00|   P183685|    6897413|          Chavigny|             Émilie|emilie.chavigny@g...|7 Rue de Sèvres, ...|     284617410|       true|     true|
| S02fq7|2024-12-29 00:00:00|   P932579|    3018520|            Dubois|            Lysiane|lysiane.dubois@or...|121 Rue Maryse Ba...|          NU

## Dataset Cegid (json)

In [117]:
df_cegid_sales = spark.read.option("multiline","true").json("Data/2024_cegid_sales.json")

df_cegid_sales.show()
df_cegid_sales.schema

+--------------------+------------------+--------------------+--------+-------------+----------------+
|               email|             price|        product_name|quantity|      sale_id|transaction_date|
+--------------------+------------------+--------------------+--------+-------------+----------------+
|                NULL|              21.8|Confiture Artisan...|       2|PA01240100001|      2024-01-05|
|isabelle.dupont@g...|              18.2|Fromage de Chèvre...|       1|PA02240100001|      2024-01-12|
|                NULL|              68.4|Huile d'Olive Ext...|       3|PA03240100001|      2024-01-18|
|                NULL|              28.5|Vin Rouge Bordeau...|       1|BO01240100001|      2024-01-25|
|thomas.lefebvre@y...|              31.5|Pâté de Campagne ...|       2|BO02240100001|      2024-01-03|
|                NULL|             14.95|Miel de Lavande d...|       1|MO01240100001|      2024-01-08|
|                NULL|                45|Terrine de Foie G...|       1|LY

StructType([StructField('email', StringType(), True), StructField('price', StringType(), True), StructField('product_name', StringType(), True), StructField('quantity', LongType(), True), StructField('sale_id', StringType(), True), StructField('transaction_date', StringType(), True)])

## Dataset product ref

In [118]:
df_product_ref = spark.read.csv("Data/2025_product_reference.csv", header=True, inferSchema=True)

df_product_ref.show()
df_product_ref.schema

+----------+--------------------+-----+-----------+
|product_id|        product_name|price|   category|
+----------+--------------------+-----+-----------+
|   P294857|Chocolat Noir 70%...| 12.5| confiserie|
|   P946283|Coffret Découvert...| 35.0|       luxe|
|   P183752|Huile d'Olive Ext...| 22.8|     divers|
|   P659104|Confiture Artisan...| 10.9| confiserie|
|   P835027|Vin Rouge Bordeau...| 28.5|        vin|
|   P317496|Fromage de Chèvre...| 18.2|    fromage|
|   P502618|Pâté de Campagne ...|15.75|charcuterie|
|   P729345|Miel de Lavande d...|14.95|     divers|
|   P461809|Terrine de Foie G...| 45.0|       luxe|
|   P210573|Assortiment de Ma...|29.99| confiserie|
|   P987654|Jambon Ibérique d...|180.0|charcuterie|
|   P345678|  Caviar d'Aquitaine|350.0|       luxe|
|   P123456|Champagne Brut Mi...| 85.0|        vin|
|   P789012|Truffes Noires du...| 95.0|       luxe|
|   P234567| Safran en Filaments| 48.0|     divers|
|   P890123|Huîtres Spéciales...| 38.5|     divers|
|   P456789|

StructType([StructField('product_id', StringType(), True), StructField('product_name', StringType(), True), StructField('price', DoubleType(), True), StructField('category', StringType(), True)])

## Dataset boutiques

In [119]:
# lecture du dataset boutique en ne prenant pas compte de la ligne header
df_boutique = spark.read.option("header", "false").option("comment", "s").csv("Data/2025_boutiques.csv", inferSchema=True, sep="|")

df_boutique.show()

+----+--------------------+--------------------+
| _c0|                 _c1|                 _c2|
+----+--------------------+--------------------+
|PA01|Épicerie Fine Par...|12 Rue des Francs...|
|PA02|Saveurs de France...|35 Avenue de la R...|
|PA03| Le Gourmet Parisien|8 Place de la Bas...|
|BO01| Délices de Bordeaux|42 Cours de l'Int...|
|BO02|Terroirs de Bordeaux|15 Rue du Parleme...|
|MO01|Le Panier Montpel...|7 Rue de la Loge,...|
|LY01|Gastronomie Lyonn...|28 Quai Saint-Ant...|
|LY02|  Les Halles de Lyon|102 Cours Lafayet...|
|MA01|Saveurs de Proven...|50 Rue Paradis, 1...|
|LI01|   Au Palais Lillois|18 Rue Basse, 598...|
|RE01|Épices et Saveurs...|3 Place du Champ ...|
|ST01|Trésors d'Alsace ...|25 Rue des Halleb...|
|CL01|   Volcans Gourmands|6 Place de Jaude,...|
+----+--------------------+--------------------+



# Alimentation du datalake

In [120]:
from pyspark.sql import SparkSession
import sqlite3
from pyspark.sql.functions import col
import pandas as pd

# Nom de la base de données SQLite
db_path = "sales_2024.db"

# 1. D'abord, convertir les DataFrames Spark en DataFrames pandas
# car la conversion directe de Spark à SQLite n'est pas native

# Table store
df_boutique_pd = df_boutique.toPandas()

# Table product
df_product_ref_pd = df_product_ref.toPandas()

# Table sales (transactions)
df_sales_cegid_pd = df_cegid_sales.toPandas()

# Table customer
df_salesforce_pd = df_sales.toPandas()

# 2. Établir une connexion avec SQLite et créer/écrire dans les tables
conn = sqlite3.connect(db_path)

# Écrire les données dans les tables, créer les tables si elles n'existent pas
df_boutique_pd.to_sql("store_lake", conn, if_exists="replace", index=False)
df_product_ref_pd.to_sql("product_lake", conn, if_exists="replace", index=False)
df_salesforce_pd.to_sql("transaction_salesforce_lake", conn, if_exists="replace", index=False)
df_sales_cegid_pd.to_sql("tansaction_cegid_lake", conn, if_exists="replace", index=False)

# Fermer la connexion
conn.close()

print(f"Les données ont été enregistrées avec succès dans {db_path}")

Les données ont été enregistrées avec succès dans sales_2024.db


# Transformation des données

In [121]:
# alimentation de la table store
df_boutique = df_boutique.toDF("store_id", "store_name", "store_address")
# a sauvegarder dans le warehouse

# alimentation de la table product
df_product_ref = df_product_ref.toDF("product_id", "name", "unit_price", "category")
# a sauvegarder dans le warehouse

### Alimentation de la table transaction

In [ ]:
from pyspark.sql.functions import col, lit, substring, coalesce

# alimentation de la table transaction
df_cegid_product = df_cegid_sales.join(df_product_ref, df_cegid_sales.product_name == df_product_ref.name, "inner")
df_cegid_sales = df_cegid_product.select("sale_id", "transaction_date", "product_id", "email", "price", "quantity")

# Créer une table de mappage email -> customer_id à partir des données Salesforce
customer_email_mapping = df_sales.select(
    "customer_email", 
    "customer_id"
).filter(
    (col("customer_email").isNotNull()) & 
    (col("customer_id").isNotNull())
).dropDuplicates(["customer_email"])

# 1. Enrichir df_cegid_sales avec les informations nécessaires et les customer_id
df_cegid_complete = df_cegid_product.join(
    customer_email_mapping,
    df_cegid_product.email == customer_email_mapping.customer_email,
    "left"  # Left join pour garder toutes les ventes CEGID
).select(
    "sale_id",
    "product_id",
    col("email").alias("customer_email"),
    # Utiliser le customer_id du mapping si disponible, sinon NULL
    coalesce(customer_email_mapping.customer_id, lit(None).cast("int")).alias("customer_id"),
    col("price").cast("double").alias("sales_price"),
    "quantity",
    "transaction_date",
    substring("sale_id", 1, 4).alias("store_id")
)

# 2. Préparer df_sales avec les mêmes colonnes et le même format
df_sales_complete = df_sales.join(
    df_product_ref,
    df_sales.product_id == df_product_ref.product_id,
    "inner"
).select(
    df_sales.sale_id,
    df_sales.product_id,
    "customer_id",
    "customer_email",
    col("unit_price").alias("sales_price"),
    lit(1).cast("int").alias("quantity"),  # Utiliser la quantité existante ou 1 par défaut
    "transaction_date",
    substring("sale_id", 1, 4).alias("store_id")
)

# 3. Maintenant, nous pouvons réaliser l'union des deux datasets
df_sales_combined = df_sales_complete.select(
    "sale_id",
    "product_id",
    "customer_id",
    "sales_price",
    "quantity", 
    "transaction_date",
    "store_id"
).union(
    df_cegid_complete.select(
        "sale_id",
        "product_id",
        "customer_id",
        "sales_price",
        "quantity",
        "transaction_date",
        "store_id"
    )
)

# 4. Vérifier les boutiques valides en faisant une jointure avec df_boutique
df_sales_combined = df_sales_combined.join(
    df_boutique, 
    df_sales_combined.store_id == df_boutique.store_id,
    "left"  # Jointure à gauche pour garder toutes les ventes
).select(
    df_sales_combined.sale_id,
    df_sales_combined.product_id,
    df_sales_combined.customer_id,
    df_sales_combined.sales_price,
    df_sales_combined.quantity,
    df_sales_combined.transaction_date,
    # Utiliser le store_id de df_boutique si existe, sinon garder celui calculé
    coalesce(df_boutique.store_id, df_sales_combined.store_id).alias("store_id")
)

# Afficher le résultat
df_sales_combined.show()

# Voir combien de lignes CEGID ont été enrichies avec un customer_id
print("Nombre de ventes CEGID avec customer_id récupéré:", 
      df_cegid_complete.filter(col("customer_id").isNotNull()).count())
print("Nombre total de ventes CEGID:", df_cegid_complete.count())

+-------+----------+-----------+-----------+--------+-------------------+--------+
|sale_id|product_id|customer_id|sales_price|quantity|   transaction_date|store_id|
+-------+----------+-----------+-----------+--------+-------------------+--------+
| S8b1c2|   P729345|    8275941|      14.95|       1|2024-01-03 00:00:00|    S8b1|
| Sc9a5d|   P317496|    3164057|       18.2|       1|2024-01-15 00:00:00|    Sc9a|
| S1f7h4|   P946283|    5480293|       35.0|       1|2024-01-22 00:00:00|    S1f7|
| S6k3j9|   P502618|    1927358|      15.75|       1|2024-01-08 00:00:00|    S6k3|
| S0pgh6|   P210573|    6701462|      29.99|       1|2024-01-29 00:00:00|    S0pg|
| Sr8m5n|   P659104|    4051729|       10.9|       1|2024-01-11 00:00:00|    Sr8m|
| S4t7y1|   P183752|    9632804|       22.8|       1|2024-01-18 00:00:00|    S4t7|
| S9v5w8|   P835027|    6719035|       28.5|       1|2024-01-05 00:00:00|    S9v5|
| Sl0x3z|   P461809|    4596280|       45.0|       1|2024-01-26 00:00:00|    Sl0x|
| S5

### Alimentation de la table customer

In [123]:
df_customer_salesforce = df_sales.select("customer_id", "customer_email").distinct()
df_customer_cegid = df_cegid_complete.select("customer_id", "customer_email").distinct()
df_customer = df_customer_salesforce.union(df_customer_cegid).distinct()

df_customer = df_customer.filter(
    (col("customer_id").isNotNull()) | 
    (col("customer_email").isNotNull())
)

print("Nombre de clients avant filtrage:", df_customer_salesforce.union(df_customer_cegid).distinct().count())
print("Nombre de clients après filtrage:", df_customer.count())
df_customer.show(10)

Nombre de clients avant filtrage: 200
Nombre de clients après filtrage: 199
+-----------+--------------------+
|customer_id|      customer_email|
+-----------+--------------------+
|    5551222| emma.paul@gmail.com|
|    6719035|camille.petit@out...|
|    5796104|tara.millos@gmail...|
|    7591313|    kb1313@gmail.com|
|    7352016| jack.smith@yahoo.fr|
|    4102657|  zara.petrov@sfr.fr|
|    3258676|thibault.giraudea...|
|    6585422| icare.ulios@free.fr|
|    1093586|manon.chevalier@y...|
|    1011622|emma.lemoine@wana...|
+-----------+--------------------+
only showing top 10 rows



## Envoi vers le datawarehouse

In [124]:
from pyspark.sql import SparkSession
import sqlite3
from pyspark.sql.functions import col
import pandas as pd

# Nom de la base de données SQLite
db_path = "sales_2024.db"

# 1. D'abord, convertir les DataFrames Spark en DataFrames pandas
# car la conversion directe de Spark à SQLite n'est pas native

# Table store
df_boutique_pd = df_boutique.toPandas()

# Table product
df_product_ref_pd = df_product_ref.toPandas()

# Table sales (transactions)
df_sales_combined_pd = df_sales_combined.toPandas()

# Table customer
df_customer_pd = df_customer.toPandas()

# 2. Établir une connexion avec SQLite et créer/écrire dans les tables
conn = sqlite3.connect(db_path)

# Écrire les données dans les tables, créer les tables si elles n'existent pas
df_boutique_pd.to_sql("store", conn, if_exists="replace", index=False)
df_product_ref_pd.to_sql("product", conn, if_exists="replace", index=False)
df_sales_combined_pd.to_sql("transaction", conn, if_exists="replace", index=False)
df_customer_pd.to_sql("customer", conn, if_exists="replace", index=False)

# Fermer la connexion
conn.close()

print(f"Les données ont été enregistrées avec succès dans {db_path}")

Les données ont été enregistrées avec succès dans sales_2024.db
